# Notebook 01 - Data wrangling and feature engineering (Objective 2)
**FIT/Foundations of Data Science - Assessment 3 - Sydney Group 19**

This notebook builds the two modelling datasets required by the brief:

| Dataset | Rows | Unit | Target | Explanatory variables |
|---|---|---|---|---|
| `lr21_match_level.csv` | 104 | one match | goal difference (Team A - Team B) | exactly 8 |
| `lr22_team_match_level.csv` | 208 | one team in one match | goals scored by that team | exactly 8 |

**Golden rule - no leakage.** Every explanatory variable is known *before kick-off*:
either fixed before the tournament (FIFA ranking of 11 Jun 2026, World Cup history,
geography, the published schedule) or computed only from matches that were already
finished on an earlier date.

**Raw sources (all in `data/raw/`)**
* `matches_raw.txt` - all 104 results incl. extra time and venue (openfootball/worldcup.json,
  cross-checked against Wikipedia's knockout-stage page and the FBref team table below).
* `team_totals_fbref.csv` - FBref team table used in Assessment 2 (used here only to **validate** the match file).
* `team_info.csv` - FIFA/Coca-Cola Men's World Ranking published 11 Jun 2026 (via MLSsoccer.com),
  previous World Cup appearances and confederation (Wikipedia, 2026 FIFA World Cup qualification), capital coordinates.
* `venues.csv` - 16 host stadiums with coordinates and altitude.

In [1]:
import numpy as np
import pandas as pd
from wc_utils import haversine_km

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
RAW, OUT = "../data/raw/", "../data/processed/"

## 1. Load and parse the 104 match results
Score used = score at the end of play (**including extra time, excluding penalty shoot-outs**),
because a shoot-out is not a goal and FIFA records such matches as draws.

In [2]:
cols = ["num", "stage", "date", "team_a", "team_b", "ft", "et", "pens", "venue"]
m = pd.read_csv(RAW + "matches_raw.txt", sep="|", names=cols, dtype=str)
m["date"] = pd.to_datetime(m["date"])
m["final_score"] = np.where(m["et"] != "-", m["et"], m["ft"])
m[["goals_a", "goals_b"]] = m["final_score"].str.split("-", expand=True).astype(int)
m["extra_time"] = (m["et"] != "-").astype(int)
m["shootout"] = (m["pens"] != "-").astype(int)
m["knockout"] = (~m["stage"].str.startswith("Group")).astype(int)

# chronological match_id 1..104 (ties on the same day keep official listing order)
m = m.reset_index().sort_values(["date", "index"]).reset_index(drop=True)
m["match_id"] = np.arange(1, len(m) + 1)
m["goal_diff"] = m["goals_a"] - m["goals_b"]
print("Matches:", len(m), "| group:", (m.knockout == 0).sum(), "| knockout:", m.knockout.sum())
print("Extra time:", m.extra_time.sum(), "| shoot-outs:", m.shootout.sum())
m[["match_id", "stage", "date", "team_a", "team_b", "goals_a", "goals_b", "venue"]].head()

Matches: 104 | group: 72 | knockout: 32
Extra time: 9 | shoot-outs: 4


,match_id,stage,date,team_a,team_b,goals_a,goals_b,venue
0,1,Group A,2026-06-11,Mexico,South Africa,2,0,Mexico City
1,2,Group A,2026-06-11,South Korea,Czech Republic,2,1,Guadalajara (Zapopan)
2,3,Group B,2026-06-12,Canada,Bosnia & Herzegovina,1,1,Toronto
3,4,Group D,2026-06-12,USA,Paraguay,4,1,Los Angeles (Inglewood)
4,5,Group B,2026-06-13,Qatar,Switzerland,1,1,San Francisco Bay Area (Santa Clara)


## 2. Data-quality checks
**Validation against an independent source:** aggregating our 104 results per team must
reproduce FBref's matches played, goals for and goals against for all 48 teams.

In [3]:
fb = pd.read_csv(RAW + "team_totals_fbref.csv")
fb["team"] = fb["Squad"].str.split(" ", n=1).str[1]
name_map = {"Bosnia–Herz": "Bosnia & Herzegovina", "Côte d'Ivoire": "Ivory Coast",
            "Congo DR": "DR Congo", "Cabo Verde": "Cape Verde", "IR Iran": "Iran",
            "Korea Republic": "South Korea", "Türkiye": "Turkey", "Czechia": "Czech Republic"}
fb["team"] = fb["team"].replace(name_map)

long_chk = pd.concat([
    m.rename(columns={"team_a": "team", "goals_a": "gf", "goals_b": "ga"})[["team", "gf", "ga"]],
    m.rename(columns={"team_b": "team", "goals_b": "gf", "goals_a": "ga"})[["team", "gf", "ga"]]])
agg = long_chk.groupby("team").agg(MP=("gf", "size"), GF=("gf", "sum"), GA=("ga", "sum")).reset_index()
chk = fb.merge(agg, on="team", suffixes=("_fbref", "_ours"))
mismatch = chk[(chk.MP_fbref != chk.MP_ours) | (chk.GF_fbref != chk.GF_ours) | (chk.GA_fbref != chk.GA_ours)]
print(f"Teams matched: {len(chk)}/48 | mismatching teams: {len(mismatch)}")
print("Total goals:", m.goals_a.sum() + m.goals_b.sum(), "| duplicated fixtures:",
      m.duplicated(["date", "team_a", "team_b"]).sum(), "| missing values:", m.isna().sum().sum())
assert len(chk) == 48 and mismatch.empty, "Match file disagrees with FBref!"

Teams matched: 48/48 | mismatching teams: 0
Total goals: 308 | duplicated fixtures: 0 | missing values: 0


## 3. Team and venue reference data

In [4]:
teams = pd.read_csv(RAW + "team_info.csv")
venues = pd.read_csv(RAW + "venues.csv")
assert set(m.team_a).union(m.team_b) == set(teams.team), "team names not harmonised"
assert set(m.venue) <= set(venues.venue), "unknown venue"
teams["strength"] = -np.log(teams["fifa_rank_2026_06_11"])   # higher = stronger
teams["elite_confed"] = teams["confederation"].isin(["UEFA", "CONMEBOL"]).astype(int)
teams.sort_values("fifa_rank_2026_06_11").head(8)

,team,fifa_rank_2026_06_11,wc_prev_appearances,confederation,host,capital,capital_lat,capital_lon,strength,elite_confed
0,Argentina,1,18,CONMEBOL,0,Buenos Aires,-34.60,-58.38,-0.000000,1
1,Spain,2,16,UEFA,0,Madrid,40.42,-3.70,-0.693147,1
2,France,3,16,UEFA,0,Paris,48.86,2.35,-1.098612,1
3,England,4,16,UEFA,0,London,51.51,-0.13,-1.386294,1
4,Portugal,5,8,UEFA,0,Lisbon,38.72,-9.14,-1.609438,1
5,Brazil,6,22,CONMEBOL,0,Brasilia,-15.79,-47.88,-1.791759,1
6,Morocco,7,6,CAF,0,Rabat,34.02,-6.84,-1.945910,0
7,Netherlands,8,11,UEFA,0,Amsterdam,52.37,4.90,-2.079442,1


**Why `-ln(rank)` instead of raw rank?** The gap between rank 1 and rank 10 matters far more
on the pitch than the gap between rank 61 and 70. A log transform makes the rank scale
roughly linear in team quality, which is what a *linear* regression needs.

## 4. Reshape to team-match ("long") format - 208 rows
Each match produces two rows, one from each team's perspective. This is the unit of analysis for
Regression 2.2 and also the easiest structure for computing each team's pre-match form.

In [5]:
base = ["match_id", "date", "stage", "knockout", "venue"]
A = m[base + ["team_a", "team_b", "goals_a", "goals_b"]].copy()
A.columns = base + ["team", "opponent", "gf", "ga"]; A["side"] = "A"
B = m[base + ["team_b", "team_a", "goals_b", "goals_a"]].copy()
B.columns = base + ["team", "opponent", "gf", "ga"]; B["side"] = "B"
tm = pd.concat([A, B]).sort_values(["team", "date", "match_id"]).reset_index(drop=True)
tm = tm.merge(teams[["team", "strength", "fifa_rank_2026_06_11", "wc_prev_appearances",
                     "elite_confed", "host", "capital_lat", "capital_lon"]], on="team")
tm = tm.merge(venues[["venue", "country", "lat", "lon", "altitude_m"]], on="venue")
tm = tm.sort_values(["team", "date", "match_id"]).reset_index(drop=True)
print(tm.shape)

(208, 21)


## 5. Pre-match features (computed ONLY from earlier matches)

* **Form, with a cold-start prior.** Before matchday 1 a team has no tournament data. Rather than
  inventing 0 goals, we shrink towards the **2022 World Cup average of 1.34 goals per team-match**
  (172 goals in 64 matches) using one pseudo-match:
  `gf_form = (goals scored so far + 1.34) / (matches so far + 1)`; likewise `ga_form`, and
  `gd_form = goal difference so far / (matches so far + 1)`.
* **Rest days** = days since the team's previous match. For matchday 1 (no previous match) we impute
  the median of all observed rest intervals; the schedule was published in advance.
* **Travel km** = great-circle distance from the team's previous venue (0 for its first match).
* **Home distance km** = distance from the team's capital to the stadium (proxy for crowd support,
  climate and time-zone familiarity).
* **Host** = 1 only when a co-host plays *in its own country* (e.g. Canada's R32 match in Los Angeles is 0).

In [6]:
PRIOR_GPM = 172 / 128          # 2022 WC goals per team-match
g = tm.groupby("team")
tm["n_prev"] = g.cumcount()
tm["gf_cum_prev"] = g["gf"].cumsum() - tm["gf"]
tm["ga_cum_prev"] = g["ga"].cumsum() - tm["ga"]
tm["gf_form"] = (tm.gf_cum_prev + PRIOR_GPM) / (tm.n_prev + 1)
tm["ga_form"] = (tm.ga_cum_prev + PRIOR_GPM) / (tm.n_prev + 1)
tm["gd_form"] = (tm.gf_cum_prev - tm.ga_cum_prev) / (tm.n_prev + 1)

tm["rest_days"] = g["date"].diff().dt.days
rest_median = tm["rest_days"].median()
tm["rest_days"] = tm["rest_days"].fillna(rest_median)
print("Median rest interval used for matchday 1:", rest_median)

prev_lat, prev_lon = g["lat"].shift(), g["lon"].shift()
tm["travel_km"] = haversine_km(prev_lat, prev_lon, tm.lat, tm.lon).fillna(0)
tm["home_dist_km"] = haversine_km(tm.capital_lat, tm.capital_lon, tm.lat, tm.lon)
host_country = {"Mexico": "Mexico", "USA": "USA", "Canada": "Canada"}
tm["host_home"] = [int(host_country.get(t) == c) for t, c in zip(tm.team, tm.country)]

# leakage self-check: features of a team's first match cannot depend on any result
first = tm[tm.n_prev == 0]
assert (first.gd_form == 0).all() and np.allclose(first.gf_form, PRIOR_GPM)
print("Leakage check passed: matchday-1 form values use only the prior.")

Median rest interval used for matchday 1: 5.0
Leakage check passed: matchday-1 form values use only the prior.


## 6. Bring the opponent's pre-match information onto each row

In [7]:
opp_cols = ["match_id", "team", "strength", "ga_form", "gd_form", "rest_days", "travel_km",
            "home_dist_km", "host_home", "wc_prev_appearances", "elite_confed"]
opp = tm[opp_cols].rename(columns={c: "opp_" + c for c in opp_cols if c != "match_id"})
tm = tm.merge(opp, left_on=["match_id", "opponent"], right_on=["match_id", "opp_team"]).drop(columns="opp_team")
assert len(tm) == 208

## 7. Dataset for Regression 2.1 - goal difference (104 rows x 8 predictors)
Because the target is *signed* (A minus B), every predictor is also a **difference A minus B**, so
swapping the two teams flips the sign of both sides of the equation consistently.

In [8]:
a = tm[tm.side == "A"].set_index("match_id").sort_index()
lr21 = pd.DataFrame({
    "match_id": a.index,
    "team_a": a.team.values, "team_b": a.opponent.values, "stage": a.stage.values,
    "strength_diff": (a.strength - a.opp_strength).values,            # ln(rank_B / rank_A)
    "form_gd_diff": (a.gd_form - a.opp_gd_form).values,
    "host_diff": (a.host_home - a.opp_host_home).values,
    "rest_diff": (a.rest_days - a.opp_rest_days).values,
    "travel_diff_1000km": ((a.travel_km - a.opp_travel_km) / 1000).values,
    "home_dist_diff_1000km": ((a.home_dist_km - a.opp_home_dist_km) / 1000).values,
    "wc_exp_diff": (a.wc_prev_appearances - a.opp_wc_prev_appearances).values,
    "elite_confed_diff": (a.elite_confed - a.opp_elite_confed).values,
    "goal_diff": (a.gf - a.ga).values,
})
FEAT21 = ["strength_diff", "form_gd_diff", "host_diff", "rest_diff", "travel_diff_1000km",
          "home_dist_diff_1000km", "wc_exp_diff", "elite_confed_diff"]
assert lr21.shape[0] == 104 and len(FEAT21) == 8
lr21.to_csv(OUT + "lr21_match_level.csv", index=False)
lr21[FEAT21 + ["goal_diff"]].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
strength_diff,104.0,0.38,1.58,-4.14,-0.78,0.43,1.42,4.20
form_gd_diff,104.0,0.14,1.16,-4.00,0.00,0.00,0.75,2.50
host_diff,104.0,0.07,0.35,-1.00,0.00,0.00,0.00,1.00
rest_diff,104.0,0.12,0.53,-1.00,0.00,0.00,0.00,3.00
travel_diff_1000km,104.0,-0.06,1.21,-4.03,-0.29,0.00,0.52,2.32
home_dist_diff_1000km,104.0,-0.74,4.21,-14.58,-3.06,-0.40,1.36,12.86
wc_exp_diff,104.0,2.74,8.68,-18.00,-3.00,2.00,10.00,21.00
elite_confed_diff,104.0,0.15,0.75,-1.00,0.00,0.00,1.00,1.00
goal_diff,104.0,0.48,2.02,-4.00,-1.00,0.00,2.00,6.00


## 8. Dataset for Regression 2.2 - goals scored by a team (208 rows x 8 predictors)

In [9]:
lr22 = pd.DataFrame({
    "row_id": np.arange(1, 209), "match_id": tm.match_id, "team": tm.team,
    "opponent": tm.opponent, "stage": tm.stage,
    "strength_gap": tm.strength - tm.opp_strength,          # SHARED construct with 2.1
    "host_home": tm.host_home,                              # SHARED
    "rest_days": tm.rest_days,                              # SHARED
    "home_dist_1000km": tm.home_dist_km / 1000,             # SHARED
    "gf_form": tm.gf_form,                                  # new: own attacking form
    "opp_ga_form": tm.opp_ga_form,                          # new: opponent's defensive form
    "altitude_km": tm.altitude_m / 1000,                    # new: venue altitude
    "knockout": tm.knockout,                                # new: stakes / tactics
    "goals": tm.gf,
}).sort_values(["match_id", "team"]).reset_index(drop=True)
lr22["row_id"] = np.arange(1, 209)
FEAT22 = ["strength_gap", "host_home", "rest_days", "home_dist_1000km",
          "gf_form", "opp_ga_form", "altitude_km", "knockout"]
assert lr22.shape[0] == 208 and len(FEAT22) == 8
lr22.to_csv(OUT + "lr22_team_match_level.csv", index=False)
lr22[FEAT22 + ["goals"]].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
strength_gap,208.0,0.00,1.63,-4.20,-1.08,0.00,1.08,4.20
host_home,208.0,0.06,0.24,0.00,0.00,0.00,0.00,1.00
rest_days,208.0,5.22,0.85,3.00,5.00,5.00,6.00,8.00
home_dist_1000km,208.0,7.90,3.05,0.01,5.99,7.95,9.89,16.68
gf_form,208.0,1.56,0.62,0.45,1.17,1.34,1.87,4.17
opp_ga_form,208.0,1.28,0.59,0.22,0.87,1.17,1.34,4.17
altitude_km,208.0,0.26,0.54,0.00,0.00,0.03,0.27,2.24
knockout,208.0,0.31,0.46,0.00,0.00,0.00,1.00,1.00
goals,208.0,1.48,1.39,0.00,0.00,1.00,2.00,7.00


## 9. Overlap rule check (brief: at most 4 shared explanatory variables)
We count overlap **conservatively at construct level**: a variable counts as shared if it carries the
same information in both models, even if 2.1 uses the A-minus-B version.

In [10]:
overlap = pd.DataFrame({
    "construct": ["Team strength (FIFA rank, log)", "Host playing at home", "Rest days",
                  "Distance from home capital", "Tournament goal-difference form",
                  "Travel since last match", "World Cup experience", "Elite confederation",
                  "Own attacking form (GF)", "Opponent defensive form (GA)", "Venue altitude",
                  "Knockout match"],
    "LR 2.1 variable": ["strength_diff", "host_diff", "rest_diff", "home_dist_diff_1000km",
                        "form_gd_diff", "travel_diff_1000km", "wc_exp_diff", "elite_confed_diff",
                        "-", "-", "-", "-"],
    "LR 2.2 variable": ["strength_gap", "host_home", "rest_days", "home_dist_1000km",
                        "-", "-", "-", "-", "gf_form", "opp_ga_form", "altitude_km", "knockout"]})
overlap["shared"] = (overlap["LR 2.1 variable"] != "-") & (overlap["LR 2.2 variable"] != "-")
print("Shared constructs:", overlap.shared.sum(), "(limit = 4)")
assert overlap.shared.sum() <= 4
overlap

Shared constructs: 4 (limit = 4)


,construct,LR 2.1 variable,LR 2.2 variable,shared
0,"Team strength (FIFA rank, log)",strength_diff,strength_gap,True
1,Host playing at home,host_diff,host_home,True
2,Rest days,rest_diff,rest_days,True
3,Distance from home capital,home_dist_diff_1000km,home_dist_1000km,True
4,Tournament goal-difference form,form_gd_diff,-,False
5,Travel since last match,travel_diff_1000km,-,False
6,World Cup experience,wc_exp_diff,-,False
7,Elite confederation,elite_confed_diff,-,False
8,Own attacking form (GF),-,gf_form,False
9,Opponent defensive form (GA),-,opp_ga_form,False


## 10. Data dictionary (saved for the repository)

In [11]:
dd = pd.DataFrame([
    ("strength_diff", "2.1", "ln(rank_B) - ln(rank_A); FIFA ranking 11 Jun 2026", "pre-tournament"),
    ("form_gd_diff", "2.1", "difference in shrunken goal-difference per game from earlier WC matches", "earlier matches"),
    ("host_diff", "2.1", "host_home(A) - host_home(B)", "schedule"),
    ("rest_diff", "2.1", "rest days A - rest days B", "schedule"),
    ("travel_diff_1000km", "2.1", "km travelled since previous match, A - B (thousands)", "schedule"),
    ("home_dist_diff_1000km", "2.1", "capital-to-stadium distance, A - B (thousands of km)", "geography"),
    ("wc_exp_diff", "2.1", "previous World Cup appearances, A - B", "pre-tournament"),
    ("elite_confed_diff", "2.1", "UEFA/CONMEBOL indicator, A - B", "pre-tournament"),
    ("goal_diff", "2.1 target", "goals A - goals B incl. extra time, excl. shoot-out", "outcome"),
    ("strength_gap", "2.2", "ln(rank_opp) - ln(rank_team)", "pre-tournament"),
    ("host_home", "2.2", "1 if co-host playing in its own country", "schedule"),
    ("rest_days", "2.2", "days since team's previous match (matchday 1 = median)", "schedule"),
    ("home_dist_1000km", "2.2", "capital-to-stadium distance (thousands of km)", "geography"),
    ("gf_form", "2.2", "(goals scored so far + 1.34) / (games so far + 1)", "earlier matches"),
    ("opp_ga_form", "2.2", "opponent's (goals conceded so far + 1.34) / (games so far + 1)", "earlier matches"),
    ("altitude_km", "2.2", "stadium altitude (km above sea level)", "venue"),
    ("knockout", "2.2", "1 if Round of 32 or later", "schedule"),
    ("goals", "2.2 target", "goals scored by the team incl. extra time", "outcome"),
], columns=["variable", "model", "definition", "known before kick-off because"])
dd.to_csv(OUT + "data_dictionary.csv", index=False)
dd

,variable,model,definition,known before kick-off because
0,strength_diff,2.1,ln(rank_B) - ln(rank_A); FIFA ranking 11 Jun 2026,pre-tournament
1,form_gd_diff,2.1,difference in shrunken goal-difference per game from earlier WC matches,earlier matches
2,host_diff,2.1,host_home(A) - host_home(B),schedule
3,rest_diff,2.1,rest days A - rest days B,schedule
4,travel_diff_1000km,2.1,"km travelled since previous match, A - B (thousands)",schedule
5,home_dist_diff_1000km,2.1,"capital-to-stadium distance, A - B (thousands of km)",geography
6,wc_exp_diff,2.1,"previous World Cup appearances, A - B",pre-tournament
7,elite_confed_diff,2.1,"UEFA/CONMEBOL indicator, A - B",pre-tournament
8,goal_diff,2.1 target,"goals A - goals B incl. extra time, excl. shoot-out",outcome
9,strength_gap,2.2,ln(rank_opp) - ln(rank_team),pre-tournament
